# Logs CPF Intergrall VF1 — Silver Layer

**Migrated from:** Alteryx workflow `Logs CPF Intergrall VF1.yxmd`
**Migration date:** 2026-08-06
**Medallion tier:** Silver (cleansed & conformed)
**Source tables:** `{bronze}.cat099_log_itgl`, `{bronze}.base_funcionarios`
**Target tables:** `{silver}.log_itgl_consultas`, `{silver}.rh_funcionarios`

## Alteryx Tool Mapping
| Step | Alteryx Tool | ToolID | SQL Operation |
|---|---|---|---|
| 1 | DateTime (`dd/MM/yyyy`) | 47 | `to_date(..., 'dd/MM/yyyy')` |
| 2 | Cleanse (UPPER `log_itgl_nm_cns`) | 58 | UPPER aplic. no Gold join |
| 3 | Formula (`/` → `-`) | 46 | `regexp_replace()` |
| 4 | Filter (`>= yesterday`) | 48 | `WHERE date_filter / date >= 2022` |
| 5 | Formula (`CPF_RET`, `CPF_RST`) | 39 | `regexp_replace()` no SELECT |
| 6 | Cleanse (UPPER `Nome`) | 59 | UPPER aplic. no Gold join |
| 7 | Filter (`CPF_RST = "1"`) | 40 | `WHERE ... RLIKE '^(DIR|MEMBRO|PRES)'` |

## Correções aplicadas após revisão do XML original

1. **Não há Unique/Duplicates** neste workflow — removido.
2. **Join (Tool 41)** usa saída **`Join`** (matched, `.yxmd:790`) = INNER JOIN.
3. **Chave do Join**: `log_itgl_nm_cns` = `Nome` (ambos UPPER).
4. **Filtro pós-join**: `Status IGI != 'DESLIGADO COMPLETAMENTE'` (Tool 57).

In [0]:
%run ./00_config

In [0]:
# Imports mantidos apenas para o data quality (célula final)
from pyspark.sql import functions as F

## 1. Log — transformação em SQL (Tools 47, 58, 46, 48)

A query Silver do log aplica:

1. **`to_date`** — converte `log_itgl_dat` de `dd/MM/yyyy` para DATE.
2. **`regexp_replace`** — normaliza data (`/` → `-`).
3. **`concat`** — monta a chave de consulta para rastreabilidade.
4. **Filtro temporal** — configurável via `${date_filter}` (recent=mês corrente, full=desde 2022).

Não há lógica de Unique/Duplicates neste workflow — todas as consultas são mantidas.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.${silver_schema}.log_itgl_consultas
COMMENT 'Silver — consultas de CPF no Intergrall com data convertida e normalização. Migrado de Logs CPF Intergrall VF1.yxmd (Tools 47/58/46/48).'
AS
SELECT
  CAST(log_itgl_seq AS BIGINT)      AS log_itgl_seq,
  log_itgl_usu,
  log_itgl_nm,
  log_itgl_nm_cns,
  log_itgl_cpf_cns,
  regexp_replace(log_itgl_dat, '/', '-') AS log_itgl_dat,
  to_date(log_itgl_dat, 'dd/MM/yyyy') AS data_convertida,
  log_itgl_hor,
  log_itgl_obs,
  log_itgl_arq,
  log_itgl_dat_itg,
  log_itgl_usu_itg,
  CAST(log_itgl_ati AS INT)         AS log_itgl_ati,
  concat(
    coalesce(log_itgl_usu, ''),
    coalesce(log_itgl_nm, ''),
    coalesce(log_itgl_nm_cns, ''),
    coalesce(log_itgl_cpf_cns, ''),
    coalesce(log_itgl_dat, '')
  ) AS chave_cons
FROM ${log_input_table}
WHERE (
  '${date_filter}' = 'full'
  OR date_trunc('MONTH', to_date(log_itgl_dat, 'dd/MM/yyyy'))
     = date_trunc('MONTH', current_date())
)
AND to_date(log_itgl_dat, 'dd/MM/yyyy') >= '2022-01-01'

## 2. RH — CPF normalizado e recorte de cargos de direção (Tools 16, 17)

* `cpf_ret` — remove `.`, `-` e zeros à esquerda. **Nunca é usado** no fluxo
  original: o join é por nome. Mantido para paridade e conferência.
* `cpf_rst` — `'1'` quando o cargo começa com `DIR`, `MEMBRO` ou `PRES`.
  O filtro mantém só esses: diretores, membros de comitê e presidência.

Colunas renomeadas para snake_case conforme as boas práticas de Silver do
skill — o que também normaliza os nomes de coluna (`Nome` → `nome_funcionario`,
`Cargo` → `cargo_nome`, `Status FA` → `status`).

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.${silver_schema}.rh_funcionarios
COMMENT 'Silver — funcionarios em cargos de direcao (DIR/MEMBRO/PRES), ativos e desligados, com CPF normalizado. Migrado de Logs CPF Intergrall VF1.yxmd (Tools 16/17).'
AS
SELECT
  Nome                          AS nome_funcionario,
  Cargo                         AS cargo_nome,
  CPF                           AS cpf,
  `Status FA`                   AS status,
  `Status IGI`                  AS status_igi,
  Empresa                       AS empresa,
  -- cpf_ret: remove pontos, traços e zeros à esquerda (paridade com Alteryx)
  regexp_replace(
    regexp_replace(
      regexp_replace(CPF, '\\.', ''),
    '-', ''),
  '^0+', '')                    AS cpf_ret
FROM `${catalog}`.${bronze_schema}.base_funcionarios
WHERE upper(coalesce(Cargo, '')) RLIKE '^(DIR|MEMBRO|PRES)'

## 3. Data quality (skill, Fase 3, passo 10)

Estatísticas via PySpark `.select()` (mantido em Python para assertions).

In [0]:
log_dq = spark.table(T_SILVER_LOG).select(
    F.count("*").alias("linhas"),
    F.coalesce(F.countDistinct("chave_cons"), F.lit(0)).alias("chaves_distintas"),
    F.coalesce(F.sum(F.when(F.col("log_itgl_usu").isNull(), 1).otherwise(0)), F.lit(0)).alias("usu_nulos"),
    F.coalesce(F.sum(F.when(F.col("data_convertida").isNull(), 1).otherwise(0)), F.lit(0)).alias("data_invalida"),
    F.min("data_convertida").alias("dt_min"),
    F.max("data_convertida").alias("dt_max"),
).collect()[0]

rh_dq = spark.table(T_SILVER_RH).select(
    F.count("*").alias("linhas"),
    F.coalesce(F.countDistinct("nome_funcionario"), F.lit(0)).alias("nomes_distintos"),
    F.coalesce(F.sum(F.when(F.col("nome_funcionario").isNull(), 1).otherwise(0)), F.lit(0)).alias("nome_nulos"),
    F.coalesce(F.sum(F.when(F.length("cpf_ret") != 11, 1).otherwise(0)), F.lit(0)).alias("cpf_fora_de_11"),
).collect()[0]

print(f"{T_SILVER_LOG}: {log_dq.linhas:,} linhas | {log_dq.chaves_distintas:,} chaves | "
      f"janela {log_dq.dt_min}..{log_dq.dt_max}")
print(f"  usu nulos={log_dq.usu_nulos:,}  datas inválidas={log_dq.data_invalida:,}")
print(f"{T_SILVER_RH}: {rh_dq.linhas:,} linhas | {rh_dq.nomes_distintos:,} nomes | "
      f"nome nulos={rh_dq.nome_nulos:,}  cpf_ret != 11 díg={rh_dq.cpf_fora_de_11:,}")

# datas não convertidas quebram o recorte silenciosamente — o formato mudou na origem
assert (log_dq.data_invalida or 0) == 0, (
    f"{log_dq.data_invalida} registros com log_itgl_dat fora de dd/MM/yyyy. "
    "O filtro de data descartaria esses registros em silêncio — verifique a origem."
)
# a chave do join é o nome; nulo do lado do RH nunca casa e infla o anti-join
assert (rh_dq.nome_nulos or 0) == 0, (
    f"{rh_dq.nome_nulos} funcionários de direção sem nome — a chave do join é o nome, "
    "então esses registros produziriam falsos positivos no anti-join."
)
if rh_dq.linhas == 0:
    print("AVISO: nenhum cargo DIR/MEMBRO/PRES no RH — o anti-join devolveria TODO o log.")

In [0]:
dbutils.notebook.exit(
    f"SILVER OK | log={log_dq.linhas} | rh_direcao={rh_dq.linhas}"
)